# Text and Vector Embeddings with Similarity Search Using Bedrock API

**Lab:** Convert text into vectors, measure semantic similarity, and build a working retrieval pipeline on Amazon Bedrock.

**Models Used:**

| Purpose | Model ID |
|---------|----------|
| Text to vector (embeddings) | `amazon.titan-embed-text-v2:0` |
| Vector to answer (generation) | `us.amazon.nova-2-lite-v1:0` |

**Region:** `us-west-2`

---

## Scenario
You are a Machine Learning Engineer at K21 Academy. Your team needs to measure how closely different pieces of text relate to each other so you can build semantic search, content recommendation, and document question-answering features. You will use Amazon Titan Text Embeddings V2 to turn text into vectors and Amazon Nova 2 Lite to turn retrieved text back into readable answers.

## What You Will Learn
- Generate text embeddings using Amazon Titan Text Embeddings V2
- Choose an embedding dimension and understand the trade-off
- Calculate cosine similarity between two pieces of text
- Build an in-memory vector store from a document set
- Perform semantic search and compare it against keyword search
- Combine retrieval with Nova 2 Lite to produce grounded answers

---

## Architecture

```
documents.txt  --->  Titan Embeddings V2  --->  vector store (NumPy array)
                                                        |
User query     --->  Titan Embeddings V2  --->  cosine similarity
                                                        |
                                                  top matches
                                                        |
                                              Nova 2 Lite  --->  answer
```

> **Two models, two jobs.** Titan turns text into numbers. Nova turns the retrieved text back into a sentence. An embedding model cannot write an answer, and a text model cannot produce an embedding.

---
## Cell 1 — Install Libraries and Create the Bedrock Client

Run this cell to install and import the required packages, then create the Bedrock Runtime client.

> **Note:** The pip install produces no visible output. If no error appears, the libraries installed correctly. ✅

In [1]:
# Install required packages
import sys
!{sys.executable} -m pip install boto3 numpy matplotlib --quiet

# Import required libraries
import json
import time
import boto3
import numpy as np

# Create the Amazon Bedrock Runtime client
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    region_name='us-west-2'
)

# Model IDs — two different models doing two different jobs
EMBED_MODEL_ID = 'amazon.titan-embed-text-v2:0'   # text  -> vector
CHAT_MODEL_ID = 'us.amazon.nova-2-lite-v1:0'      # text  -> text

# Embedding settings
# Titan Text Embeddings V2 supports 256, 512 or 1024 dimensions (default 1024)
DIMENSIONS = 1024
NORMALIZE = True   # unit-length vectors make cosine similarity a simple dot product

print('✅ Libraries imported and Bedrock client created successfully.')
print(f'   Embedding model : {EMBED_MODEL_ID}')
print(f'   Chat model      : {CHAT_MODEL_ID}')
print(f'   Dimensions      : {DIMENSIONS}')

✅ Libraries imported and Bedrock client created successfully.
   Embedding model : amazon.titan-embed-text-v2:0
   Chat model      : us.amazon.nova-2-lite-v1:0
   Dimensions      : 1024


---
## Cell 2 — Create the Embedding Helper and Generate Your First Vector

An embedding is a list of floating-point numbers representing the **meaning** of a piece of text. Similar meanings produce similar numbers, even when the words are completely different.

This cell defines a reusable `embed()` function and runs it on one sentence so you can see the output.

**Request format for Titan Text Embeddings V2:**
```
{ "inputText": string, "dimensions": int, "normalize": boolean }
```

In [2]:
def embed(text, dimensions=DIMENSIONS, normalize=NORMALIZE):
    """Convert a string into a vector using Titan Text Embeddings V2."""
    body = json.dumps({
        'inputText': text,
        'dimensions': dimensions,
        'normalize': normalize
    })

    response = bedrock_runtime.invoke_model(
        modelId=EMBED_MODEL_ID,
        body=body,
        contentType='application/json',
        accept='application/json'
    )

    payload = json.loads(response['body'].read())
    return np.array(payload['embedding']), payload['inputTextTokenCount']


# Generate the first embedding
sample_text = 'Amazon Bedrock provides access to foundation models through a single API.'
vector, token_count = embed(sample_text)

print('=' * 60)
print('         YOUR FIRST EMBEDDING')
print('=' * 60)
print(f'Input text    : {sample_text}')
print(f'Token count   : {token_count}')
print(f'Vector shape  : {vector.shape}')
print(f'First 5 values: {np.round(vector[:5], 4)}')
print(f'Last 5 values : {np.round(vector[-5:], 4)}')
print(f'Vector length : {np.linalg.norm(vector):.4f}  (1.0 because normalize=True)')
print('=' * 60)
print('✅ One sentence became 1024 numbers.')

         YOUR FIRST EMBEDDING
Input text    : Amazon Bedrock provides access to foundation models through a single API.
Token count   : 14
Vector shape  : (1024,)
First 5 values: [-0.1059  0.0142  0.0164 -0.0364 -0.0536]
Last 5 values : [ 0.0224  0.0168  0.0095 -0.0221 -0.0145]
Vector length : 1.0000  (1.0 because normalize=True)
✅ One sentence became 1024 numbers.


---
## Cell 3 — Compare Embedding Dimensions

Titan V2 lets you choose 256, 512 or 1024 dimensions. Fewer dimensions means less storage and faster search, at some cost to accuracy.

This is a real design decision when building a vector store, so it is worth seeing the difference rather than accepting the default blindly.

In [3]:
text_a = 'How do I reset my account password?'
text_b = 'I forgot my login credentials and need to regain access.'

print(f'Text A: {text_a}')
print(f'Text B: {text_b}')
print()
print(f"{'Dimensions':<14}{'Similarity':<14}{'Bytes per vector':<18}")
print('-' * 46)

for dims in [256, 512, 1024]:
    vec_a, _ = embed(text_a, dimensions=dims)
    vec_b, _ = embed(text_b, dimensions=dims)
    similarity = float(np.dot(vec_a, vec_b))
    print(f'{dims:<14}{similarity:<14.4f}{dims * 4:<18}')

print()
print('✅ All three sizes agree the sentences are related,')
print('   even though they share almost no words in common.')

Text A: How do I reset my account password?
Text B: I forgot my login credentials and need to regain access.

Dimensions    Similarity    Bytes per vector  
----------------------------------------------
256           0.5122        1024              
512           0.4327        2048              
1024          0.4160        4096              

✅ All three sizes agree the sentences are related,
   even though they share almost no words in common.


---
## Cell 4 — Measure Similarity with Cosine Similarity

Cosine similarity measures the angle between two vectors, ignoring their length.

- **1.0** — identical meaning
- **around 0.0** — unrelated
- **negative** — opposing meaning (rare with text embeddings)

Because `normalize=True` gives every vector length 1, cosine similarity reduces to a dot product. The full formula is written out below so you can see what is actually being calculated.

In [4]:
def cosine_similarity(vec_1, vec_2):
    """Cosine similarity between two vectors. 1.0 means identical meaning."""
    return float(np.dot(vec_1, vec_2) / (np.linalg.norm(vec_1) * np.linalg.norm(vec_2)))


sentence_pairs = [
    ('The server is down', 'The system is currently unavailable'),
    ('The server is down', 'I would like to order a pizza'),
    ('How much does shipping cost?', 'What are your delivery charges?'),
    ('How much does shipping cost?', 'How long does delivery take?'),
    ('cat', 'kitten'),
    ('cat', 'automobile')
]

print(f"{'Text A':<32}{'Text B':<38}{'Score':<8}")
print('-' * 78)

for text_1, text_2 in sentence_pairs:
    v1, _ = embed(text_1)
    v2, _ = embed(text_2)
    score = cosine_similarity(v1, v2)
    print(f'{text_1[:30]:<32}{text_2[:36]:<38}{score:<8.4f}')

print()
print('✅ Compare rows 3 and 4 — both mention shipping,')
print('   but only one asks about cost. The embedding captures that difference.')

Text A                          Text B                                Score   
------------------------------------------------------------------------------
The server is down              The system is currently unavailable   0.4527  
The server is down              I would like to order a pizza         0.1546  
How much does shipping cost?    What are your delivery charges?       0.3706  
How much does shipping cost?    How long does delivery take?          0.1556  
cat                             kitten                                0.4376  
cat                             automobile                            0.1855  

✅ Compare rows 3 and 4 — both mention shipping,
   but only one asks about cost. The embedding captures that difference.


---
## Cell 5 — Load the Knowledge Base

This cell reads `documents.txt` and treats each non-empty line as one document.

> **Note:** Make sure `documents.txt` is in the same folder as this notebook.

In [5]:
with open('documents.txt') as f:
    documents = [line.strip() for line in f if line.strip()]

print('=' * 70)
print('         KNOWLEDGE BASE LOADED')
print('=' * 70)

for index, document in enumerate(documents):
    print(f'[{index}] {document[:85]}')

print('=' * 70)
print(f'✅ {len(documents)} documents loaded successfully.')

         KNOWLEDGE BASE LOADED
[0] The price of product number 12345 is 10$
[1] The price of product number 3241 is 40$
[2] The price of product number 7123 is 13$
[3] The price of product number 9823 is 33$
[4] The price of product number 1982 is 63$
[5] The price of product number 5723 is 76$
✅ 6 documents loaded successfully.


---
## Cell 6 — Build the Vector Store

Each document is embedded once and stored in a NumPy array. In production these vectors would live in a service such as Amazon OpenSearch Serverless; here an in-memory array is enough to demonstrate the mechanics.

> **Key point:** Embedding happens once, at ingestion. Every query afterwards is just arithmetic against these stored vectors, which is why retrieval is fast.

In [6]:
print('Embedding the knowledge base...')
start_time = time.time()

corpus_vectors = np.vstack([embed(document)[0] for document in documents])

elapsed = time.time() - start_time

print()
print(f'✅ Vector store built in {elapsed:.2f} seconds.')
print(f'   Shape : {corpus_vectors.shape}')
print(f'   That is {corpus_vectors.shape[0]} documents x {corpus_vectors.shape[1]} dimensions.')
print()
print('   Each row is one document, stored as a point in 1024-dimensional space.')

Embedding the knowledge base...

✅ Vector store built in 1.04 seconds.
   Shape : (6, 1024)
   That is 6 documents x 1024 dimensions.

   Each row is one document, stored as a point in 1024-dimensional space.


---
## Cell 7 — Semantic Search

To search, the query is embedded with the **same model** and compared against every stored vector.

> **Important:** Queries and documents must use the same embedding model. Different models produce vectors in different coordinate spaces, and comparing across them gives meaningless results.

In [7]:
def semantic_search(query, top_k=3):
    """Return the top_k most similar documents with their similarity scores."""
    query_vector, _ = embed(query)
    scores = corpus_vectors @ query_vector
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    return [(documents[i], float(scores[i])) for i in ranked_indices]


test_queries = [
    'What is the price of product number 5723?',
    'How much does item 5723 cost?',
    'Tell me about the cheapest product',
    'What is the price of iPhone 15?'
]

for query in test_queries:
    print('=' * 70)
    print(f'Query: {query}')
    print('-' * 70)
    for rank, (document, score) in enumerate(semantic_search(query), start=1):
        print(f'  {rank}. [{score:.4f}] {document[:58]}')
    print()

print('✅ Notice that queries 1 and 2 return the same top result,')
print('   despite using completely different wording.')

Query: What is the price of product number 5723?
----------------------------------------------------------------------
  1. [0.7835] The price of product number 5723 is 76$
  2. [0.3948] The price of product number 9823 is 33$
  3. [0.3837] The price of product number 7123 is 13$

Query: How much does item 5723 cost?
----------------------------------------------------------------------
  1. [0.6929] The price of product number 5723 is 76$
  2. [0.3479] The price of product number 9823 is 33$
  3. [0.3181] The price of product number 7123 is 13$

Query: Tell me about the cheapest product
----------------------------------------------------------------------
  1. [0.2677] The price of product number 12345 is 10$
  2. [0.2381] The price of product number 7123 is 13$
  3. [0.2347] The price of product number 3241 is 40$

Query: What is the price of iPhone 15?
----------------------------------------------------------------------
  1. [0.1577] The price of product number 5723 is 76$
  2. 

---
## Cell 8 — Keyword Search vs Semantic Search

This is the question worth answering before adopting embeddings anywhere: why not just match keywords?

Keyword search matches **characters**. Semantic search matches **meaning**. The cell below runs both against queries that deliberately avoid the exact wording used in the documents.

In [ ]:
def keyword_search(query, top_k=3):
    """Naive keyword-overlap search, included only for comparison."""
    query_words = set(query.lower().split())
    scored = [(d, len(query_words & set(d.lower().split()))) for d in documents]
    scored.sort(key=lambda item: item[1], reverse=True)
    return scored[:top_k]


comparison_queries = [
    'How much do I pay for item 5723?',
    'What is the least expensive option?'
]

for query in comparison_queries:
    print('=' * 70)
    print(f'Query: {query}')
    print('=' * 70)

    print('  KEYWORD SEARCH (matches characters)')
    for document, overlap in keyword_search(query):
        print(f'    [{overlap} words matched] {document[:52]}')

    print('  SEMANTIC SEARCH (matches meaning)')
    for document, score in semantic_search(query):
        print(f'    [score {score:.4f}]      {document[:52]}')
    print()

print('✅ Keyword search fails when the user does not use your exact vocabulary.')

---
## Cell 9 — Visualise the Similarity Matrix

Comparing every document against every other document produces a matrix. Bright cells are semantically related pairs.

The diagonal is always 1.00, because every document is identical to itself.

In [ ]:
import matplotlib.pyplot as plt

similarity_matrix = corpus_vectors @ corpus_vectors.T

fig, ax = plt.subplots(figsize=(8, 6.5))
image = ax.imshow(similarity_matrix, cmap='viridis', vmin=0, vmax=1)

labels = [f'Doc {i}' for i in range(len(documents))]
ax.set_xticks(range(len(documents)))
ax.set_yticks(range(len(documents)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_title('Document similarity matrix (cosine)')

for i in range(len(documents)):
    for j in range(len(documents)):
        value = similarity_matrix[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center',
                color='white' if value < 0.6 else 'black', fontsize=8)

fig.colorbar(image, ax=ax, label='Cosine similarity')
plt.tight_layout()
plt.show()

print('✅ Similarity matrix generated successfully.')

---
## Cell 10 — Retrieval-Augmented Generation with Nova 2 Lite

Semantic search returns raw document lines. Useful, but not a sentence a customer would want to read.

Here both models work together: Titan finds the relevant documents, then Nova 2 Lite writes a natural-language answer grounded in **only** those documents.

> This is retrieval-augmented generation (RAG) — the same pattern Amazon Bedrock Knowledge Bases implements as a managed service.

In [ ]:
def generate_answer(query, top_k=3, show_context=True):
    """Retrieve relevant documents, then ask Nova 2 Lite to answer from them."""
    retrieved = semantic_search(query, top_k=top_k)
    context = '\n'.join(document for document, _ in retrieved)

    if show_context:
        print('Retrieved context:')
        for document, score in retrieved:
            print(f'  [{score:.4f}] {document[:60]}')
        print()

    instruction = (
        'Use only the context below to answer the question. '
        'If the answer is not in the context, say I do not know.'
    )

    user_prompt = (
        f'{instruction}\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {query}\n\n'
        'Answer:'
    )

    request_body = {
        'messages': [
            {'role': 'user', 'content': [{'text': user_prompt}]}
        ],
        'system': [
            {'text': 'You are a precise assistant. Answer only from the provided context. Never invent details.'}
        ],
        'inferenceConfig': {
            'maxTokens': 300,
            'temperature': 0.1
        }
    }

    response = bedrock_runtime.invoke_model(
        modelId=CHAT_MODEL_ID,
        body=json.dumps(request_body),
        contentType='application/json',
        accept='application/json'
    )

    payload = json.loads(response['body'].read())
    return payload['output']['message']['content'][0]['text'].strip()


demo_queries = [
    'What is the price of product number 5723?',
    'What is the price of iPhone 15?'
]

for query in demo_queries:
    print('=' * 70)
    print(f'Question: {query}')
    print('-' * 70)
    print(f'Answer  : {generate_answer(query)}')
    print()

print('✅ The second question is answered with I do not know,')
print('   because that information is not in the knowledge base.')

---
## Cell 11 — Interactive Question and Answer

Run this cell and ask your own questions.

Watch the retrieved context as well as the answer. When an answer looks wrong, the cause is almost always that the right document was never retrieved, not that the model reasoned badly.

> Type `quit` to exit the loop.

In [ ]:
print('Ask a question about the knowledge base. Type quit to exit.')
print()

while True:
    query = input('Enter your query (or quit to exit): ').strip()

    if query.lower() == 'quit':
        print('Exiting. Goodbye!')
        break

    if not query:
        continue

    try:
        print('-' * 70)
        answer = generate_answer(query)
        print(f'Answer: {answer}')
        print('-' * 70)
    except Exception as error:
        print(f'Error: {error}')

---
## Summary

In this lab, you have:

| Step | What You Did |
|------|--------------|
| Cell 1 | Installed libraries and created the Bedrock Runtime client |
| Cell 2 | Built the `embed()` helper and generated your first 1024-dimension vector |
| Cell 3 | Compared 256, 512 and 1024 dimensions and saw the storage trade-off |
| Cell 4 | Calculated cosine similarity across six sentence pairs |
| Cell 5 | Loaded `documents.txt` as the knowledge base |
| Cell 6 | Embedded the corpus once into an in-memory vector store |
| Cell 7 | Ran semantic search and ranked documents by meaning |
| Cell 8 | Compared keyword search against semantic search |
| Cell 9 | Visualised the document similarity matrix |
| Cell 10 | Combined retrieval with Nova 2 Lite to generate grounded answers |
| Cell 11 | Queried the pipeline interactively |

---

### The Key Idea

The embedding model and the generation model are **different tools**.

- Embeddings decide **what the model sees** — retrieval quality is set at embedding time
- Generation decides **how it reads** — the model can only work with what retrieval gave it

If an answer is wrong, check the retrieved context before blaming the model.

---

**Models Used:** Amazon Titan Text Embeddings V2 (`amazon.titan-embed-text-v2:0`) and Amazon Nova 2 Lite (`us.amazon.nova-2-lite-v1:0`)  
**Region:** us-west-2 (Oregon)  

> ⚠️ **Remember:** Stop and delete your SageMaker JupyterLab space, user profile and domain after completing this lab to avoid unnecessary charges.